# Oficina: Como Criar um Sniffer com `l2tap_sniffer`

Bem-vindo(a) à oficina de criação de sniffers com a biblioteca `l2tap_sniffer`.

Nesta aula, vamos entender como usar a biblioteca existente do projeto para montar sniffers em nível 2 da pilha Ethernet, com foco na placa **LILYGO T-ETH-Lite**.

Ao final da oficina, a ideia é que você consiga:

- entender a arquitetura da biblioteca
- configurar filtros, parser, runtime e backend Ethernet
- registrar callbacks para receber os frames
- montar um sniffer próprio a partir do exemplo atual do projeto


## 1. Pré-requisitos

Antes de começar, o ideal é ter:

- noções básicas de C
- noções básicas de ESP-IDF
- uma placa **LILYGO T-ETH-Lite**
- um cabo USB para programar a placa
- um cabo de rede Ethernet
- o ambiente ESP-IDF já instalado
- acesso ao terminal ESP-IDF ou VS Code com a extensão configurada

Esta oficina é voltada para iniciantes. Por isso, além do código, vamos revisar alguns conceitos importantes do caminho.


## 2. Conceitos Fundamentais

### Ethernet

Ethernet é a tecnologia de rede usada para transmitir quadros entre dispositivos em uma rede cabeada.

### Quadro Ethernet

O quadro Ethernet é a unidade de dados transmitida na rede. Ele contém, entre outras coisas:

- MAC de origem
- MAC de destino
- EtherType
- payload

### EtherType

O EtherType informa qual tipo de conteúdo está sendo transportado no quadro.

Exemplos:

- `0x0800`: IPv4
- `0x0806`: ARP
- `0x8100`: VLAN
- `0x88A4`: EtherCAT
- `0x8892`: Profinet

### L2 TAP

`L2 TAP` é o mecanismo do ESP-IDF para acessar quadros Ethernet no nível 2. Em vez de ler apenas pacotes IP já tratados pela pilha de rede, a aplicação pode ler o quadro Ethernet de forma mais crua.

### NVS

`NVS` é a área de armazenamento persistente do ESP32. Ela guarda dados do sistema e da aplicação que precisam sobreviver a reinicializações.

### VFS

`VFS` é a camada de sistema de arquivos virtual do ESP-IDF. É ela que permite acessar `/dev/net/tap` usando funções como `open()`, `read()` e `ioctl()`.

### Callback

Callback é uma função que a aplicação entrega para a biblioteca. Quando algo acontece, a biblioteca chama essa função.

### Handle

Handle é a referência da instância criada da biblioteca. Você usa esse handle para iniciar, parar e destruir o sniffer.

### Parser

O parser é a parte que pega o frame bruto e tenta interpretá-lo: Ethernet, ARP, IPv4, TCP, UDP, VLAN e assim por diante.

### Backend Ethernet

O backend Ethernet é a parte responsável por fazer o bring-up da rede, configurar pinos, PHY, clock e iniciar o driver Ethernet.


## 3. Visão Geral da Arquitetura

A arquitetura atual do projeto pode ser entendida em quatro partes:

1. **Aplicação hospedeira**
   A aplicação prepara o ambiente do ESP-IDF, monta a configuração e consome os callbacks.

2. **Biblioteca `l2tap_sniffer`**
   A biblioteca organiza o runtime, os filtros, a captura e a entrega dos dados.

3. **Backend Ethernet**
   O backend sobe a interface Ethernet com base nas configurações passadas.

4. **Parser e callbacks**
   O parser interpreta os frames e os callbacks entregam os dados para a aplicação.

Fluxo resumido:

- a aplicação inicializa `NVS`, `esp_netif`, event loop e `ESP-VFS L2 TAP`
- a aplicação monta `l2tap_sniffer_config_t`
- a biblioteca cria a instância
- a biblioteca sobe Ethernet
- a biblioteca abre os filtros L2 TAP
- a biblioteca cria as tasks de captura e análise
- a aplicação recebe os dados via callbacks


### Arquivos de configuracao e build do projeto

Antes de olhar o codigo do sniffer, vale entender cinco arquivos importantes do projeto ESP-IDF:

- `sdkconfig.defaults`: arquivo manual e versionado com os defaults desejados do projeto
- `sdkconfig`: arquivo gerado automaticamente pelo ESP-IDF com a configuracao final resolvida
- `CMakeLists.txt` : inicializa o projeto no CMake e conecta o build ao ESP-IDF
- `main/CMakeLists.txt`: registra os arquivos `.c` do app principal e suas dependencias

Regra pratica:

- `sdkconfig.defaults` diz como o projeto prefere nascer
- `sdkconfig` mostra como ele realmente nasceu depois do processamento do ESP-IDF
- os `CMakeLists.txt` dizem o que precisa ser compilado e como os componentes se conectam

### O que fica em `sdkconfig.defaults`

Neste projeto, esse arquivo deve guardar apenas opcoes estaveis de hardware e build, por exemplo:

- suporte Ethernet interno
- PHY `RTL8201`
- interface fisica `RMII`
- ativacao do `ESP-NETIF L2 TAP`
- tamanho de flash da placa

Exemplo:

```ini
CONFIG_ETHERNET_INTERNAL_SUPPORT=y
CONFIG_ETHERNET_PHY_RTL8201=y
CONFIG_ETHERNET_PHY_INTERFACE_RMII=y
CONFIG_ESP_NETIF_L2_TAP=y
CONFIG_ESPTOOLPY_FLASHSIZE="16MB"
```

Ja os pinos e alguns parametros do backend Ethernet podem ficar no codigo da aplicacao, como no exemplo minimo da oficina:

- `config.eth.mdc_gpio = 23`
- `config.eth.mdio_gpio = 18`
- `config.eth.phy_addr = 0`
- `config.eth.rmii_clk_gpio = 0`

### O que cada `CMakeLists.txt` faz

- o `CMakeLists.txt` da raiz define o projeto e inicializa o build do ESP-IDF
- o `main/CMakeLists.txt` lista `SRCS`, `INCLUDE_DIRS` e `REQUIRES` do app principal

### Fluxo recomendado

1. ajustar o `sdkconfig.defaults` com os defaults que devem ser versionados
2. revisar o `CMakeLists.txt` da raiz para garantir que o projeto ESP-IDF esta sendo inicializado corretamente
3. revisar o `main/CMakeLists.txt` sempre que adicionar ou remover arquivos `.c`
4. rodar `idf.py build` para regenerar o `sdkconfig` e validar o build

### Erros comuns

- editar `sdkconfig` manualmente como se ele fosse a fonte principal da configuracao
- esquecer um arquivo `.c` dentro de `SRCS`
- usar um header de outro componente sem declarar a dependencia em `REQUIRES`
- colocar no `sdkconfig.defaults` opcoes demais, como se ele fosse uma copia completa do `sdkconfig`


## 4. Conhecendo a API Pública

Os arquivos públicos da biblioteca são:

- `components/l2tap_sniffer/include/l2tap_sniffer.h`
- `components/l2tap_sniffer/include/l2tap_sniffer_types.h`

Principais funções:

- `l2tap_sniffer_config_default()`
- `l2tap_sniffer_create()`
- `l2tap_sniffer_start()`
- `l2tap_sniffer_stop()`
- `l2tap_sniffer_destroy()`

O tipo mais importante do ponto de vista da aplicação é:

- `l2tap_sniffer_handle_t`

Ele representa a instância criada da biblioteca.


In [ ]:
#include "l2tap_sniffer.h"

l2tap_sniffer_runtime_config_t l2tap_sniffer_runtime_config_default(void);
l2tap_sniffer_parser_config_t l2tap_sniffer_parser_config_default(void);
l2tap_sniffer_esp32_eth_config_t l2tap_sniffer_esp32_eth_config_default(void);
l2tap_sniffer_config_t l2tap_sniffer_config_default(void);

esp_err_t l2tap_sniffer_create(const l2tap_sniffer_config_t *cfg,
                              l2tap_sniffer_handle_t *out_handle);
esp_err_t l2tap_sniffer_start(l2tap_sniffer_handle_t handle);
esp_err_t l2tap_sniffer_stop(l2tap_sniffer_handle_t handle);
void l2tap_sniffer_destroy(l2tap_sniffer_handle_t handle);


## 5. Como Configurar a Biblioteca

A estrutura principal de configuração é:

- `l2tap_sniffer_config_t`

Ela agrupa:

- `interface_name`
- `filters`
- `runtime`
- `parser`
- `eth`
- `callbacks`

A forma recomendada de começar é sempre com os defaults:


In [ ]:
l2tap_sniffer_config_t config = l2tap_sniffer_config_default();


### 5.1 Filtros

Os filtros dizem quais EtherTypes serão capturados em `/dev/net/tap`.

Cada filtro tem:

- `label`: nome amigável do filtro
- `ethertype`: EtherType capturado

Exemplo para um sniffer simples de IPv4 e ARP:


In [ ]:
static const l2tap_sniffer_filter_t filters[] = {
    { "ipv4", L2TAP_SNIFFER_ETH_TYPE_IPV4 },
    { "arp", L2TAP_SNIFFER_ETH_TYPE_ARP },
};

config.filters = filters;
config.filter_count = 2;


### 5.2 Parser

O bloco `parser` define quanto a biblioteca deve interpretar dos frames.

Campos principais:

- `parse_arp`
- `parse_ipv4`
- `parse_transport`
- `detect_industrial_protocols`
- `payload_preview_bytes`

Exemplo de configuração completa do parser:


In [ ]:
config.parser.parse_arp = true;
config.parser.parse_ipv4 = true;
config.parser.parse_transport = true;
config.parser.detect_industrial_protocols = true;
config.parser.payload_preview_bytes = 32;


### 5.3 Runtime

O bloco `runtime` controla buffers e temporização interna.

Campos principais:

- `max_frame_len`
- `ring_buffer_size`
- `stats_period_ms`

Exemplo:


In [ ]:
config.runtime.max_frame_len = 1600;
config.runtime.ring_buffer_size = 16384;
config.runtime.stats_period_ms = 5000;


### 5.4 Backend Ethernet

O bloco `eth` define como a biblioteca vai subir a interface Ethernet.

Na oficina, vamos usar a configuração da **T-ETH-Lite**, baseada em:

- `RTL8201`
- `RMII`
- `power_gpio = 12`
- `mdc_gpio = 23`
- `mdio_gpio = 18`
- `phy_addr = 0`
- `rmii_clk_gpio = 0`
- `phy_reset_gpio = -1`

Exemplo:


In [ ]:
config.eth.power_gpio = 12;
config.eth.mdc_gpio = 23;
config.eth.mdio_gpio = 18;
config.eth.phy_addr = 0;
config.eth.rmii_clk_gpio = 0;
config.eth.phy_reset_gpio = -1;
config.eth.power_up_delay_ms = 100;
config.eth.link_timeout_ms = 15000;
config.eth.phy = L2TAP_SNIFFER_PHY_RTL8201;


### 5.5 Callbacks

Os callbacks são o ponto de integração da biblioteca com a aplicação.

A biblioteca pode chamar:

- `on_raw_frame`
- `on_parsed_frame`
- `on_event`
- `on_error`

Na maioria dos sniffers, o callback mais usado é `on_parsed_frame`, porque ele já entrega o frame interpretado.


In [ ]:
static void on_parsed_frame(l2tap_sniffer_handle_t handle,
                            const l2tap_sniffer_raw_frame_t *raw_frame,
                            const l2tap_sniffer_parsed_frame_t *parsed_frame,
                            void *user_ctx)
{
    (void)handle;
    (void)user_ctx;

    printf("capture=%s len=%u ethertype=0x%04x\n",
           raw_frame->capture_label,
           raw_frame->frame_len,
           parsed_frame->ethertype);
}

config.callbacks.on_parsed_frame = on_parsed_frame;


## 6. Exemplo Mínimo

Agora vamos juntar o essencial para ter um sniffer mínimo.

Esse exemplo:

- inicializa o ambiente do ESP-IDF
- define dois filtros
- configura Ethernet para a T-ETH-Lite
- registra um callback simples
- cria e inicia a biblioteca


In [ ]:
#include "esp_event.h"
#include "esp_netif.h"
#include "esp_vfs_l2tap.h"
#include "l2tap_sniffer.h"
#include "nvs_flash.h"

static void on_frame(l2tap_sniffer_handle_t handle,
                     const l2tap_sniffer_raw_frame_t *raw_frame,
                     const l2tap_sniffer_parsed_frame_t *parsed_frame,
                     void *user_ctx)
{
    (void)handle;
    (void)user_ctx;

    printf("%s -> 0x%04x\n", raw_frame->capture_label, parsed_frame->ethertype);
}

void app_main(void)
{
    static const l2tap_sniffer_filter_t filters[] = {
        { "ipv4", L2TAP_SNIFFER_ETH_TYPE_IPV4 },
        { "arp", L2TAP_SNIFFER_ETH_TYPE_ARP },
    };

    l2tap_sniffer_handle_t sniffer = NULL;
    l2tap_sniffer_config_t config = l2tap_sniffer_config_default();

    ESP_ERROR_CHECK(nvs_flash_init());
    ESP_ERROR_CHECK(esp_netif_init());
    ESP_ERROR_CHECK(esp_event_loop_create_default());
    ESP_ERROR_CHECK(esp_vfs_l2tap_intf_register(NULL));

    config.filters = filters;
    config.filter_count = 2;

    config.eth.power_gpio = 12;
    config.eth.mdc_gpio = 23;
    config.eth.mdio_gpio = 18;
    config.eth.phy_addr = 0;
    config.eth.rmii_clk_gpio = 0;
    config.eth.phy_reset_gpio = -1;
    config.eth.phy = L2TAP_SNIFFER_PHY_RTL8201;

    config.callbacks.on_parsed_frame = on_frame;

    ESP_ERROR_CHECK(l2tap_sniffer_create(&config, &sniffer));
    ESP_ERROR_CHECK(l2tap_sniffer_start(sniffer));
}


## 7. Exemplo Completo da T-ETH-Lite

A seguir está uma versão didática baseada no exemplo real do projeto. Ela usa os seis filtros atuais:

- IPv4
- ARP
- VLAN
- QinQ
- Profinet
- EtherCAT

Ela também mostra a ideia de saída em JSON, como no exemplo atual da aplicação.


In [ ]:
static const l2tap_sniffer_filter_t filters[] = {
    { "ipv4", L2TAP_SNIFFER_ETH_TYPE_IPV4 },
    { "arp", L2TAP_SNIFFER_ETH_TYPE_ARP },
    { "vlan", L2TAP_SNIFFER_ETH_TYPE_VLAN },
    { "qinq", L2TAP_SNIFFER_ETH_TYPE_QINQ },
    { "profinet", L2TAP_SNIFFER_ETH_TYPE_PROFINET },
    { "ethercat", L2TAP_SNIFFER_ETH_TYPE_ETHERCAT },
};

static void on_json_frame(l2tap_sniffer_handle_t handle,
                          const l2tap_sniffer_raw_frame_t *raw_frame,
                          const l2tap_sniffer_parsed_frame_t *parsed_frame,
                          void *user_ctx)
{
    (void)handle;
    (void)user_ctx;

    printf("{\"capture\":\"%s\",\"ethertype\":\"0x%04x\",\"src\":\"%s\",\"dst\":\"%s\"}\n",
           raw_frame->capture_label,
           parsed_frame->ethertype,
           parsed_frame->src_mac,
           parsed_frame->dst_mac);
}

void app_main(void)
{
    l2tap_sniffer_handle_t sniffer = NULL;
    l2tap_sniffer_config_t config = l2tap_sniffer_config_default();

    ESP_ERROR_CHECK(nvs_flash_init());
    ESP_ERROR_CHECK(esp_netif_init());
    ESP_ERROR_CHECK(esp_event_loop_create_default());
    ESP_ERROR_CHECK(esp_vfs_l2tap_intf_register(NULL));

    config.filters = filters;
    config.filter_count = sizeof(filters) / sizeof(filters[0]);

    config.runtime.max_frame_len = 1600;
    config.runtime.ring_buffer_size = 16384;
    config.runtime.stats_period_ms = 5000;

    config.parser.parse_arp = true;
    config.parser.parse_ipv4 = true;
    config.parser.parse_transport = true;
    config.parser.detect_industrial_protocols = true;
    config.parser.payload_preview_bytes = 32;

    config.eth.power_gpio = 12;
    config.eth.mdc_gpio = 23;
    config.eth.mdio_gpio = 18;
    config.eth.phy_addr = 0;
    config.eth.rmii_clk_gpio = 0;
    config.eth.phy_reset_gpio = -1;
    config.eth.power_up_delay_ms = 100;
    config.eth.link_timeout_ms = 15000;
    config.eth.phy = L2TAP_SNIFFER_PHY_RTL8201;

    config.callbacks.on_parsed_frame = on_json_frame;

    ESP_ERROR_CHECK(l2tap_sniffer_create(&config, &sniffer));
    ESP_ERROR_CHECK(l2tap_sniffer_start(sniffer));
}


## 8. Entendendo os Callbacks

### `on_raw_frame`

Use quando você quer olhar os bytes crus do quadro.

### `on_parsed_frame`

Use quando você quer consumir os campos já interpretados. Esse costuma ser o callback principal da maioria dos sniffers.

### `on_event`

Serve para receber eventos como:

- `L2TAP_SNIFFER_EVENT_ETH_STARTED`
- `L2TAP_SNIFFER_EVENT_LINK_UP`
- `L2TAP_SNIFFER_EVENT_IP_ACQUIRED`
- `L2TAP_SNIFFER_EVENT_CAPTURE_STARTED`
- `L2TAP_SNIFFER_EVENT_STOPPED`
- `L2TAP_SNIFFER_EVENT_ERROR`

### `on_error`

Use para integrar logs, tratamento de erro e diagnósticos.


## 9. Como Criar o Seu Próprio Sniffer

A receita prática é:

1. escolher os EtherTypes que fazem sentido para o seu caso
2. decidir se você quer frame bruto ou frame parseado
3. configurar o nível de parsing
4. escolher o formato de saída
5. iniciar a captura

### Cenário A: sniffer básico de IPv4 e ARP

- filtros: IPv4 e ARP
- callback principal: `on_parsed_frame`
- parser: completo ou quase completo
- saída: `printf` simples ou JSON

### Cenário B: sniffer industrial

- filtros: IPv4, VLAN, QinQ, Profinet, EtherCAT
- parser: com `detect_industrial_protocols = true`
- saída: JSON estruturado
- callback principal: `on_parsed_frame`


## 10. Como Ler os Dados Recebidos

A biblioteca entrega dois tipos principais:

### `l2tap_sniffer_raw_frame_t`

Campos mais usados:

- `capture_label`
- `ts_us`
- `source_filter`
- `frame_len`
- `frame[]`

### `l2tap_sniffer_parsed_frame_t`

Campos mais usados no dia a dia:

- `src_mac`
- `dst_mac`
- `outer_ethertype`
- `ethertype`
- `vlan_id`
- `has_arp`
- `has_ipv4`
- `ip_src`
- `ip_dst`
- `transport`
- `src_port`
- `dst_port`
- `payload_len`
- `payload_preview`
- `industrial`


## 11. Erros Comuns e Troubleshooting

### O link Ethernet não sobe

Verifique:

- alimentação do PHY
- pinos `MDC` e `MDIO`
- clock RMII
- endereço do PHY
- cabo de rede

### Há link, mas não há IP

Isso não impede a captura L2. A biblioteca pode continuar capturando quadros Ethernet mesmo sem DHCP.

### `filter_count` maior que `CONFIG_ESP_NETIF_L2_TAP_MAX_FDS`

Se você configurar filtros demais, a criação pode falhar. O número de filtros precisa respeitar o limite do projeto.

### Falha ao abrir `/dev/net/tap`

Verifique se a aplicação chamou:

- `esp_netif_init()`
- `esp_event_loop_create_default()`
- `esp_vfs_l2tap_intf_register(NULL)`

### Configuração errada de pinos

Se os pinos não corresponderem à placa real, o backend Ethernet pode não subir corretamente.


## 12. Limitações Atuais

Hoje a biblioteca está centrada em:

- `ESP32`
- `EMAC` interno
- `RMII`
- `RTL8201`
- exemplo principal com `T-ETH-Lite`

Ou seja:

- ela ainda não é uma solução genérica pronta para `ESP32-S2`
- ela ainda não é uma solução genérica pronta para `ESP32-S3`
- ela ainda não tem backend separado para Ethernet via `SPI`


## 13. Próximos Passos

Depois desta oficina, você pode evoluir seu sniffer de várias formas:

- mudar os EtherTypes monitorados
- trocar o callback de saída
- salvar os frames em arquivo
- enviar os dados para outro sistema
- criar relatórios a partir do callback
- preparar a biblioteca para novos backends Ethernet

O mais importante é entender que a biblioteca já separa bem:

- infraestrutura de captura
- parsing
- entrega para a aplicação

Então o seu sniffer passa a ser, principalmente, uma questão de:

- escolher a captura
- escolher o processamento
- escolher a saída


## 14. Encerramento

Nesta oficina você viu:

- o que é a biblioteca `l2tap_sniffer`
- como ela se encaixa no ESP-IDF
- como configurar filtros, parser, runtime, Ethernet e callbacks
- como montar um sniffer mínimo
- como montar um sniffer mais completo para a T-ETH-Lite

A partir daqui, você já tem base para criar sniffers próprios em cima da biblioteca atual do projeto.
